In [1]:
import altair as alt    
import polars as pl 
import upath

In [2]:
paths = tuple(upath.UPath(f's3://aind-scratch-data/ben.hardcastle/num_threads_benchmarks').iterdir())
len(paths)

148

In [3]:
paths = tuple(upath.UPath(f's3://aind-scratch-data/ben.hardcastle/num_threads_benchmarks').glob('*'))

for p in paths:
    lf = pl.scan_parquet(p.as_posix())
    if lf.collect_schema().get('OMP_THREAD_LIMIT') != pl.UInt16:
        print(p)
        lf.cast(
            {
            'MKL_NUM_THREADS': pl.UInt16,
            'OMP_THREAD_LIMIT': pl.UInt16,
            var: pl.UInt16,
            }
        ).collect().write_parquet(p.as_posix())    

s3://aind-scratch-data/ben.hardcastle/num_threads_benchmarks/072b3b31-c2d6-4b1e-b8c0-0ccf5c9a4189.parquet
s3://aind-scratch-data/ben.hardcastle/num_threads_benchmarks/13c3b1e5-e816-4da4-9ffe-e2d24f426721.parquet
s3://aind-scratch-data/ben.hardcastle/num_threads_benchmarks/1415b7e3-3c6f-473d-b3bd-37fa4a78e0f1.parquet
s3://aind-scratch-data/ben.hardcastle/num_threads_benchmarks/14504f37-a316-45cf-ac60-9113d0c99c7d.parquet
s3://aind-scratch-data/ben.hardcastle/num_threads_benchmarks/1840253b-cfc9-42bf-ae58-43fe3f30b591.parquet
s3://aind-scratch-data/ben.hardcastle/num_threads_benchmarks/1aeade70-ba71-4f9c-81bd-437e226d8975.parquet
s3://aind-scratch-data/ben.hardcastle/num_threads_benchmarks/1b824b1a-0c0b-423d-871f-e1fe111bb1f8.parquet
s3://aind-scratch-data/ben.hardcastle/num_threads_benchmarks/209092d2-ebc8-4730-90fc-11cfea617b74.parquet
s3://aind-scratch-data/ben.hardcastle/num_threads_benchmarks/226209a9-d95e-452f-b210-f76e8761e61c.parquet
s3://aind-scratch-data/ben.hardcastle/num_thre

In [2]:
df = (
    pl.scan_parquet(f's3://aind-scratch-data/ben.hardcastle/num_threads_benchmarks/')
    .filter(
        pl.col('benchmark') == 'dot',
    )
    .collect()
)

In [8]:
var = 'OMP_NUM_THREADS'

---
### multithreading in `BLAS` (Basic Linear Algebra Subprograms) can greatly affect the performance of basic matrix operations in `numpy`
- number of threads can be set via `OPENBLAS_NUM_THREADS`
- `null` below is the default, with no environment variable set
- benchmark uses `np.dot()` on two square matrices

In [9]:
(
    df
    .plot.line(
        x=f'{var}',
        y=alt.Y('wall time:Q', title='wall time (s)'),
        color='co cpu count:N',
        column='matrix size:O',
        # detail='os cpu threads:O',
        row='is_pipeline',
    )
    .resolve_scale(
        x='independent',
        y='independent',
    )
    .properties(
        title=alt.TitleParams(
            text='multithreading in BLAS affects numpy performance for larger matrices',
            subtitle=[
                "- benchmark uses `np.dot()` on two square matrices",
                "- number of threads are set via environment variable",
                "- upper lim is number of CPUs reported by the OS with nproc",   
                "- `null` is with no environment variable set",
            ],
            anchor='start',
            orient='bottom',
            offset=20,
        ),
        height=200,
        width=200,
    )
)

alt.Chart(...)

- for larger matrices, performance with no setting can be worse than setting manually 
- guidance in documentation is to set N-threads <= N-logical cores 

---

**BLAS sets the default N-threads to the number of logical cores available**

- in a 1 core / 8 GB flex machine, the OS gives misleading information about how many
  logical cores are available, which BLAS uses to set its threadpool size:

```
>>> python -m threadpoolctl -i numpy scipy.linalg
[
  {
    "user_api": "blas",
    "internal_api": "openblas",
    "num_threads": 16,
    "prefix": "libscipy_openblas",
    "filepath": "/opt/conda/lib/python3.12/site-packages/numpy.libs/libscipy_openblas64_-6bb31eeb.so",
    "version": "0.3.28",
    "threading_layer": "pthreads",
    "architecture": "SkylakeX"
  },
  {
    "user_api": "blas",
    "internal_api": "openblas",
    "num_threads": 16,
    "prefix": "libscipy_openblas",
    "filepath": "/opt/conda/lib/python3.12/site-packages/scipy.libs/libscipy_openblas-68440149.so",
    "version": "0.3.28",
    "threading_layer": "pthreads",
    "architecture": "SkylakeX"
  }
]
```

The OS misreports CPU cores across machine size, capsules and pipelines (used `os.cpu_count()`
before discovering `threadpoolctl` function):


In [ ]:
(
    df.plot.point(
        x=alt.X('co cpu count:O').title('cores requested'),
        y=alt.Y('os cpu count:O').title('os.cpu_count()'),
        facet='is_pipeline',
    )
    .properties(
        title=alt.TitleParams(
            text='CPU count reported by Ubuntu does not change with size of machine requested in CO',
            anchor='start',
            orient='bottom',
            offset=20,
            fontSize=12,
        ),
    )
)

alt.Chart(...)

In [14]:
(
    df
    .filter(
        pl.col(var).is_null(), # no explicit setting
    )
    .plot.point(
        x=alt.X('co cpu count:O').title('cores requested'),
        y=alt.Y('blas threads:O'),
        facet='is_pipeline',
    )
    .properties(
        title=alt.TitleParams(
            text='CPU count reported by Ubuntu does not change with size of machine requested in CO',
            anchor='start',
            orient='bottom',
            offset=20,
            fontSize=12,
        ),
    )
)

alt.Chart(...)

---

In [18]:
var = 'OMP_THREAD_LIMIT'
(
    df
    .filter(
        pl.col(var).is_not_null(), # explicit setting
    )
    .plot.point(
        x=f'{var}:O',
        y=alt.Y('blas threads:O').scale(reverse=True),
    )
)

alt.Chart(...)

In [12]:
def norm_time()  -> pl.Expr:
    over_cols = ['is_pipeline', 'benchmark', 'co cpu count', 'os cpu count', 'matrix size']
    # normalize wall time to value when nthreads == num cpu cores available
    return (pl.col('wall time') / pl.col('wall time').filter(pl.col(var) == pl.col('co cpu count')).first()).over(over_cols)

wall_time_chart = (
    df
    .with_columns(
        norm_time().alias('normalized wall time'),
    )
    .filter(
        pl.col('matrix size') >= 1000,
    )
    .plot.rect(
        x=f'{var}',
        y=alt.Y('co cpu count:O').scale(reverse=True),
        color=alt.Color('wall time:Q'),
        column='matrix size:O',
        detail='os cpu count:O',
        row='is_pipeline',
    )
)
(
    wall_time_chart
    .resolve_scale(
        x='independent',
        color='independent',
    )
)

alt.Chart(...)

normalizing to the wall time when N-threads == N-cores requested:

In [18]:
(
    wall_time_chart
    .encode(
        color=alt.Color('normalized wall time:Q').scale(type='log', scheme="redblue", domainMin=0.1, domainMid=1, domainMax=10, reverse=True),
    )
    .resolve_scale(
        x='independent',
    )
)

alt.Chart(...)

- setting N-threads = N-cores requested in CO generally does not make performance worse
    - NOTE: for 32/64 cores requested in pipeline, `CO_CPUS` method does not work

- the exception is pipelines (not sure whether 1 requested cpu == 1 logical or 1 physical core)

---
setting N-threads == N-cores requested makes operations faster in *most* cases:

In [20]:
(
    df
    .with_columns(
        norm_time().alias('normalized wall time'),
    )
    .filter(
        (pl.col('normalized wall time') == 1) | (pl.col(var).is_null())
    )
    .with_columns(
        pl.when(pl.col(var).is_null())
        .then(pl.lit('not set'))
        .when(pl.col('normalized wall time') == 1)
        .then(pl.lit('CO_CPUS'))
        .otherwise(pl.col(var).cast(pl.Utf8))
        .alias(var)
    )
    .filter(
        ~pl.col('normalized wall time').is_infinite(),
    )
    .plot.line(
        x=alt.X('OMP_NUM_THREADS:N').scale(reverse=True),
        y=alt.Y('normalized wall time:Q').scale(type='log'),
        color='co cpu count:N',
        row='is_pipeline',
        column='matrix size:O',
    )
    .properties(
        width=100,
        height=150,
    )
)

alt.Chart(...)

## questions
- does OPENMP setting work (recommended by numpy, possibly more general across OSs)?
- why OPENBLAS_NUM_THREADS = 8 on a 1 core machine in AWS Batch 5x faster than OPENBLAS_NUM_THREADS
  = 1
- which other libraries are being used and when (BLAS, OPENMP, MKL)?